> 在线讲义：[第 9 章 · Q、K、V 分别是什么](https://wikieden.github.io/transformer-visual/lessons/09-qkv.html) · 本书《从零读懂 Transformer》

# 第 9 章 · Q、K、V 分别是什么

配套讲义：[在线版](https://wikieden.github.io/transformer-visual/lessons/09-qkv.html) · 仓库里 `transformer-deep-dive/notes/lessons/09-qkv.md`

**怎么用**：从上往下，一格一格按 **Shift + Enter** 运行。文字格是讲解，代码格是对应的 torch 写法；数字随便改，改完再按一次 Shift + Enter 看结果怎么变。不需要 GPU。

In [ ]:
import torch

torch.set_printoptions(precision=3, sci_mode=False)  # 打印保留 3 位小数，不用科学计数法
print("torch", torch.__version__)

## 讲义 ↔ 代码

先看这张表：讲义里的每一步，对应下面哪一行代码、跑出来是什么。

| 讲义 | 公式 / 说法 | PyTorch 写法 | 跑出来 |
|---|---|---|---|
| Q 比对 K | 需求 · 每本书的标签 | `K @ q` | [3.0, 0.0, 0.6] |
| 分比例 | softmax | `torch.softmax(scores, -1)` | 菜谱 0.877、历史 0.044、编程 0.080 |
| 带走 V | 按比例取内容 | `p @ V` | [8.77, 0.44, 0.80]：主要是菜谱知识 |
| 换掉 V | 标签不变 | `V.flip(0)` | 比例不变，带走的东西变了 |
| 换掉 K | 内容不变 | `K.flip(0)` | 比例变了 |

图书馆找书：3 本书，标签 K 和内容 V 是两张不同的表（数字是编的）。

## 1. 先把要用的东西定义好

这几格只是定义，运行后没有输出也正常。

In [ ]:
import torch
BOOKS = ["家常菜谱", "世界历史", "编程入门"]
K = torch.tensor([[1.0, 0.0], [0.0, 1.0], [0.2, 0.1]])
V = torch.tensor([[10.0, 0.0, 0.0], [0.0, 10.0, 0.0], [0.0, 0.0, 10.0]])
Q = torch.tensor([3.0, 0.0])

**函数 `attend`**：三步：q 和每个 k 点积打分 → softmax 分比例 → 按比例取 v。

In [ ]:
def attend(q: torch.Tensor, k: torch.Tensor, v: torch.Tensor) -> tuple[torch.Tensor, torch.Tensor]:
    """三步：q 和每个 k 点积打分 → softmax 分比例 → 按比例取 v。"""
    scores = k @ q
    p = torch.softmax(scores, dim=-1)
    return p, p @ v

**函数 `r`**：小工具，把结果整理成好读的样子再打印。

In [ ]:
def r(t: torch.Tensor, n: int = 3) -> list[float]:
    return [round(float(x), n) for x in t]

## 2. 一步一步跑

每格对应讲义里的一步，打印的文字说明了在算什么。

In [ ]:
p, out = attend(Q, K, V)
print("分数 K @ q :", r(K @ Q))
print("比例      :", dict(zip(BOOKS, r(p))))
print("带走的内容:", r(out, 2), "（主要是菜谱知识）")

In [ ]:
print("\n换掉内容 V、标签 K 不动：比例不变，带走的东西变了")
v2 = V.flip(0)
p2, out2 = attend(Q, K, v2)
print("  比例:", r(p2), "  带走:", r(out2, 2))
print("换掉标签 K、内容 V 不动：比例变了")
k2 = K.flip(0)
p3, _ = attend(Q, k2, V)
print("  比例:", r(p3))
print("→ 比对只看 K，取走的是 V；所以 K 和 V 要分开学。")

## 3. 代码里才看得到的

1. **K 决定"看谁"，V 决定"拿什么"**：只换 V，比例一位都不变；只换 K，比例跟着变。所以 K 和 V 要分成两张表、各学各的。
2. **三个参数的顺序 (q, k, v)**：PyTorch 自带的 `F.scaled_dot_product_attention(q, k, v)` 也是这个顺序（第 12 章会用到）。

## 4. 自己改一改

- 把 `Q` 改成 [0, 3]（想找历史书），比例和带走的内容怎么变？

在下面的空格子里写，或者直接改上面的格子。

In [ ]:
# 在这里写你的代码

## 口诀

> Q 去问，K 来答"像不像"，V 是最后拿走的东西；
> 比对用标签，带走的是内容；每个词三个都有，既找别人，也被别人找。